#### Importing required libraries 

In [2]:
import pandas as pd
from utils import Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
import warnings
warnings.simplefilter("ignore")

In [4]:
train_dataset = 'charlie_hebdo'
test_dataset = 'ottawashooting'
time_cut =3*60*24
processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
           test_dataset, time_cut=time_cut,test_size=0.7)

processor.load_data()
processor.process_data()
train,test = processor.get_final_dataframes()


rumour
1    307
0    293
Name: count, dtype: int64


In [5]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_test =test['rumour']

#### Example  training

In [7]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd


# Calculate class imbalance ratio (optional)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    scale_pos_weight=pos_weight,      
    learning_rate=0.01,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
# Train the model
model.fit(
    X_train,
    y_train,
    eval_metric=["binary_logloss", "auc"]
)


LGBMClassifier(bagging_fraction=0.8, bagging_freq=5, feature_fraction=0.8,
               learning_rate=0.01, n_estimators=200, n_jobs=-1,
               objective='binary', random_state=42,
               scale_pos_weight=np.float64(2.7839195979899496), verbose=-1)

In [8]:
y_train_prob = model.predict_proba(X_train)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
evaluate(y_train, y_train_pred, y_train_prob, label="Train")
evaluate(y_test, y_test_pred, y_test_prob, label="Test")

  - Accuracy:  0.9783
  - Precision: 0.9708
  - Recall:    0.9464
  - AUC:       0.9977

  - Accuracy:  0.7133
  - Precision: 0.8902
  - Recall:    0.5016
  - AUC:       0.8508



#### Setting MLflow Experiment

In [9]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
#mlflow.set_experiment("spyder-experiment")
import mlflow.pytorch
mlflow.set_experiment("Light Gbm  2025-11-04 Ottawa Shooting TF")

2025/11/09 18:25:41 INFO mlflow.tracking.fluent: Experiment with name 'Light Gbm  2025-11-04 Ottawa Shooting TF' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/95', creation_time=1762712741693, experiment_id='95', last_update_time=1762712741693, lifecycle_stage='active', name='Light Gbm  2025-11-04 Ottawa Shooting TF', tags={}>

#### Loading dataset statistics to get the final time cut 

In [11]:
df_posts_by_time_cut = pd.read_csv('ottawa_shooting_posts_by_time_cut.csv')

In [12]:
time_cut_last_post = int(df_posts_by_time_cut[df_posts_by_time_cut.post==\
                         int(df_posts_by_time_cut['post'].max())].time_cut.min())

In [13]:
time_cut_last_post

599

* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [14]:
previous_node_count = 0

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\n=== Time Cut: {time_cut} ===")
    
    train_dataset = 'charlie_hebdo'
    #test_dataset = 'sydneysiege'
    test_dataset = 'ottawashooting'
    #test_dataset = 'germanwings_crash'
    #test_dataset = 'ferguson'
    time_cut =time_cut
    processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
               test_dataset, time_cut=time_cut,test_size=0.7)
    
    processor.load_data()
    processor.process_data()
    train,test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        scale_pos_weight=pos_weight,
        learning_rate=0.01,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train,
                eval_metric=["binary_logloss", "auc"]
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 40 ===
rumour
0    62
1    57
Name: count, dtype: int64
New Instances: 119

=== Time Cut: 70 ===
rumour
0    95
1    81
Name: count, dtype: int64
New Instances: 57

=== Time Cut: 100 ===
rumour
0    117
1    108
Name: count, dtype: int64
New Instances: 49

=== Time Cut: 130 ===


KeyboardInterrupt: 